# PEAK M-ATH — Process Memory Allocation Anomaly Scoring

**Ref:** M26 — Process Memory Allocation Anomaly Scoring  
**M-ATH approach:** Unsupervised anomaly detection (Isolation Forest) and density clustering (DBSCAN) on memory allocation features to identify stealthy implants, hollowing, and remote threads.

Detect process memory injection techniques such as Process Hollowing, DLL Injection, and reflective DLL loading by isolating processes that exhibit anomalous memory allocation profiles. By monitoring dynamic memory adjustments on endpoints, threat hunters can uncover stealthy implants, packing, and beaconing code that bypass traditional on-disk antivirus checks.

## How to use
1. Put process memory allocation CSV exports into `input/`
2. Run all cells
3. Review prioritized findings in `output/`

In [ ]:
# Scenario mode: anchor paths to this notebook's scenario folder.
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / 'detection_logics').exists() and (cur / 'scenarios').exists():
            return cur
        cur = cur.parent
    raise RuntimeError('Unable to locate repository root from current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
SCENARIO_DIR = REPO_ROOT / 'scenarios' / 'process_memory_allocation_anomaly_scoring'
if not SCENARIO_DIR.exists():
    raise FileNotFoundError(f'Scenario folder not found: {SCENARIO_DIR.relative_to(REPO_ROOT)}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)  # keep imports stable

INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True
print(f'Scenario folder: {SCENARIO_DIR.relative_to(REPO_ROOT)}')

In [ ]:
import glob
import math
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 20)
print("Libraries loaded successfully.")

## PREPARE — Plan your Approach

- **Select Topic:** Process Memory Allocation Anomaly Scoring (MITRE ATT&CK [T1055](https://attack.mitre.org/techniques/T1055/) Process Injection).
- **Research Topic:** Adversaries allocate and write payload code directly into process memory to bypass on-disk detection. The allocation typically involves API calls like `VirtualAllocEx` or `VirtualProtectEx` with execution rights (e.g. `PAGE_EXECUTE_READWRITE`). Remote thread creation via `CreateRemoteThread` or thread context hijacking executes the payload.
- **Identify Datasets:** Telemetry of process memory allocation APIs from EDR logs (e.g. SentinelOne `Process Modification`, Defender `VirtualAllocApiCall`, Sysmon Event 8/10).
- **Select Algorithms:** Unsupervised Isolation Forest for outlier detection scoring combined with DBSCAN clustering to profile baseline activity.

In [ ]:
# Helper to generate high-quality synthetic process memory allocation logs
def generate_synthetic_data(num_records=1000):
    np.random.seed(42)
    
    # Normal processes do VirtualAlloc for JIT, heap management, DLL loading, etc.
    normal_apps = [
        ("C:\\Windows\\System32\\svchost.exe", "C:\\Windows\\System32\\svchost.exe", "VirtualAllocApiCall", (4, 64), "PAGE_READWRITE", False, 0.4),
        ("C:\\Windows\\System32\\svchost.exe", "C:\\Windows\\System32\\svchost.exe", "VirtualProtectApiCall", (4, 32), "PAGE_READONLY", False, 0.2),
        ("C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe", "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe", "VirtualAllocApiCall", (64, 4096), "PAGE_EXECUTE_READWRITE", False, 0.2),
        ("C:\\Windows\\System32\\explorer.exe", "C:\\Windows\\System32\\explorer.exe", "VirtualAllocApiCall", (4, 256), "PAGE_READWRITE", False, 0.1),
        ("C:\\Windows\\System32\\explorer.exe", "C:\\Windows\\System32\\explorer.exe", "VirtualProtectApiCall", (4, 128), "PAGE_EXECUTE_READ", False, 0.05),
        ("C:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE", "C:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE", "VirtualAllocApiCall", (4, 128), "PAGE_READWRITE", False, 0.05)
    ]
    
    records = []
    
    # Generate background noise
    for _ in range(num_records):
        idx = np.random.choice(len(normal_apps), p=[x[6] for x in normal_apps])
        app = normal_apps[idx]
        
        computer = np.random.choice(["HOST-01", "HOST-02", "HOST-03", "HOST-04", "HOST-05"])
        # Random timestamp within last 24h
        timestamp = pd.Timestamp.now() - pd.to_timedelta(np.random.randint(1, 86400), unit='s')
        size = int(np.random.uniform(app[3][0], app[3][1]) * 1024)
        
        records.append({
            "Computer": computer,
            "Timestamp": timestamp.isoformat(),
            "InitiatingProcessFileName": app[0],
            "TargetProcessFileName": app[1],
            "ActionType": app[2],
            "RequestedSize": size,
            "RequestedProtect": app[4],
            "MemoryState": "MEM_COMMIT",
            "ThreadStartAddress": "0x0",
            "UnmappedAddress": app[5],
            "CallerModule": "C:\\Windows\\System32\\ntdll.dll" if "svchost" in app[0] else app[0],
            "IsTargetSystemProcess": "System32" in app[1]
        })
        
    # Add specific anomalous/malicious injections
    # Attack 1: Process Hollowing (update.exe -> svchost.exe remote alloc RWX + thread)
    records.append({
        "Computer": "HOST-01",
        "Timestamp": (pd.Timestamp.now() - pd.to_timedelta(500, unit='s')).isoformat(),
        "InitiatingProcessFileName": "C:\\Users\\user1\\AppData\\Local\\Temp\\update.exe",
        "TargetProcessFileName": "C:\\Windows\\System32\\svchost.exe",
        "ActionType": "VirtualAllocApiCall",
        "RequestedSize": 1048576, # 1MB
        "RequestedProtect": "PAGE_EXECUTE_READWRITE", # RWX
        "MemoryState": "MEM_COMMIT",
        "ThreadStartAddress": "0x0",
        "UnmappedAddress": True,
        "CallerModule": "Unknown",
        "IsTargetSystemProcess": True
    })
    records.append({
        "Computer": "HOST-01",
        "Timestamp": (pd.Timestamp.now() - pd.to_timedelta(498, unit='s')).isoformat(),
        "InitiatingProcessFileName": "C:\\Users\\user1\\AppData\\Local\\Temp\\update.exe",
        "TargetProcessFileName": "C:\\Windows\\System32\\svchost.exe",
        "ActionType": "CreateRemoteThreadApiCall",
        "RequestedSize": 0,
        "RequestedProtect": "PAGE_EXECUTE_READ",
        "MemoryState": "MEM_COMMIT",
        "ThreadStartAddress": "0x20f12000000", # Unmapped address
        "UnmappedAddress": True,
        "CallerModule": "Unknown",
        "IsTargetSystemProcess": True
    })
    
    # Attack 2: DLL Injection (word.exe -> explorer.exe)
    records.append({
        "Computer": "HOST-02",
        "Timestamp": (pd.Timestamp.now() - pd.to_timedelta(1200, unit='s')).isoformat(),
        "InitiatingProcessFileName": "C:\\Program Files\\Microsoft Office\\root\\Office16\\WINWORD.EXE",
        "TargetProcessFileName": "C:\\Windows\\System32\\explorer.exe",
        "ActionType": "VirtualAllocApiCall",
        "RequestedSize": 65536,
        "RequestedProtect": "PAGE_EXECUTE_READWRITE",
        "MemoryState": "MEM_COMMIT",
        "ThreadStartAddress": "0x0",
        "UnmappedAddress": True,
        "CallerModule": "Unknown",
        "IsTargetSystemProcess": True
    })
    
    # Attack 3: Reflective DLL Injection (rundll32.exe self injection RWX + unmapped thread)
    records.append({
        "Computer": "HOST-03",
        "Timestamp": (pd.Timestamp.now() - pd.to_timedelta(1800, unit='s')).isoformat(),
        "InitiatingProcessFileName": "C:\\Windows\\System32\\rundll32.exe",
        "TargetProcessFileName": "C:\\Windows\\System32\\rundll32.exe",
        "ActionType": "VirtualAllocApiCall",
        "RequestedSize": 262144,
        "RequestedProtect": "PAGE_EXECUTE_READWRITE",
        "MemoryState": "MEM_COMMIT",
        "ThreadStartAddress": "0x0",
        "UnmappedAddress": True,
        "CallerModule": "Unknown",
        "IsTargetSystemProcess": True
    })
    records.append({
        "Computer": "HOST-03",
        "Timestamp": (pd.Timestamp.now() - pd.to_timedelta(1798, unit='s')).isoformat(),
        "InitiatingProcessFileName": "C:\\Windows\\System32\\rundll32.exe",
        "TargetProcessFileName": "C:\\Windows\\System32\\rundll32.exe",
        "ActionType": "CreateRemoteThreadApiCall",
        "RequestedSize": 0,
        "RequestedProtect": "PAGE_EXECUTE_READ",
        "MemoryState": "MEM_COMMIT",
        "ThreadStartAddress": "0x1a8f8000000",
        "UnmappedAddress": True,
        "CallerModule": "Unknown",
        "IsTargetSystemProcess": True
    })

    # Attack 4: Remote injection into LSASS
    records.append({
        "Computer": "HOST-04",
        "Timestamp": (pd.Timestamp.now() - pd.to_timedelta(2500, unit='s')).isoformat(),
        "InitiatingProcessFileName": "C:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe",
        "TargetProcessFileName": "C:\\Windows\\System32\\lsass.exe",
        "ActionType": "VirtualAllocApiCall",
        "RequestedSize": 524288,
        "RequestedProtect": "PAGE_EXECUTE_READWRITE",
        "MemoryState": "MEM_COMMIT",
        "ThreadStartAddress": "0x0",
        "UnmappedAddress": True,
        "CallerModule": "Unknown",
        "IsTargetSystemProcess": True
    })
    
    return pd.DataFrame(records)

In [ ]:
# Load data from input folder or fall back to generating synthetic data
input_pattern = INPUT_DIR / "*.csv"
input_files = sorted(glob.glob(str(input_pattern)))

if input_files:
    print(f"Loading data from {len(input_files)} CSV files...")
    dfs = []
    for path in input_files:
        try:
            dfs.append(pd.read_csv(path))
        except Exception as e:
            print(f"Error reading {path}: {e}")
    df = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df):,} records.")
else:
    print("No CSV files found in input directory. Generating synthetic dataset...")
    df = generate_synthetic_data()
    # Save the synthetic dataset to input folder for future use/repeatability
    sample_file = INPUT_DIR / "sample_memory_allocations.csv"
    df.to_csv(sample_file, index=False)
    print(f"Generated and saved {len(df):,} synthetic records to {sample_file.name}")

df.head()

## EXECUTE — Feature Engineering and Anomaly Modeling

To detect memory anomalies, we will extract relevant features from the memory allocations:
1. **Numeric Features**:
   - `LogSize`: `log10(RequestedSize + 1)` because allocation sizes span multiple orders of magnitude.
2. **Boolean/Categorical Features**:
   - `IsRWX`: Page protection contains execution and write rights (e.g. `PAGE_EXECUTE_READWRITE`).
   - `IsRX`: Page protection contains execution but not write rights (e.g. `PAGE_EXECUTE_READ`).
   - `IsRemote`: The initiating process allocates memory in a different target process (`InitiatingProcessFileName` != `TargetProcessFileName`).
   - `IsUnmappedStart`: Thread start address / allocation is in unmapped memory.
   - `IsTargetSystem`: Target process is a key system process (resides in `System32` or contains `lsass`, `svchost`).
   - `ProcessFleetFrequency`: Rarity of the initiating process across the fleet. We count how many times this process file name appears and take its reciprocal.

In [ ]:
def extract_features(data):
    df_feat = data.copy()
    
    # 1. Base clean paths
    df_feat['InitProc'] = df_feat['InitiatingProcessFileName'].fillna('').str.split('\\').str[-1].str.lower()
    df_feat['TargetProc'] = df_feat['TargetProcessFileName'].fillna('').str.split('\\').str[-1].str.lower()
    
    # 2. Page protection features
    df_feat['IsRWX'] = df_feat['RequestedProtect'].fillna('').str.contains('EXECUTE_READWRITE|EXECUTE_WRITE', case=False, regex=True).astype(int)
    df_feat['IsRX'] = df_feat['RequestedProtect'].fillna('').str.contains('EXECUTE_READ', case=False, regex=True).astype(int)
    
    # 3. Target process features
    df_feat['IsRemote'] = (df_feat['InitProc'] != df_feat['TargetProc']).astype(int)
    df_feat['IsTargetSystem'] = df_feat['IsTargetSystemProcess'].astype(int)
    
    # 4. Unmapped memory execution
    df_feat['IsUnmappedStart'] = df_feat['UnmappedAddress'].astype(int)
    
    # 5. Log size
    df_feat['LogSize'] = df_feat['RequestedSize'].apply(lambda x: math.log10(max(x, 0) + 1))
    
    # 6. Process Fleet Frequency (rarity)
    freq = df_feat['InitProc'].value_counts()
    df_feat['FleetCount'] = df_feat['InitProc'].map(freq)
    # Rare processes get a higher score
    df_feat['FleetRarity'] = 1.0 / (df_feat['FleetCount'] + 0.1)
    
    # 7. ActionType indicator
    df_feat['IsRemoteThread'] = df_feat['ActionType'].str.contains('CreateRemoteThread|Thread', case=False).astype(int)
    
    return df_feat

df_features = extract_features(df)
df_features.head()

In [ ]:
# Select features for unsupervised machine learning
features_for_ml = ['IsRWX', 'IsRX', 'IsRemote', 'IsTargetSystem', 'IsUnmappedStart', 'LogSize', 'FleetRarity', 'IsRemoteThread']

X = df_features[features_for_ml].copy()

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")

### 1. Isolation Forest for Outlier Scoring

The Isolation Forest algorithm isolates observations by randomly selecting a feature and then randomly selecting a split value between the maximum and minimum values of the selected feature. Outliers require fewer splits to be isolated.

In [ ]:
# Train Isolation Forest
iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
df_features['anomaly_score_iforest'] = -iso.fit_predict(X_scaled)  # map outlier to 1, inlier to -1
df_features['iforest_raw_score'] = -iso.score_samples(X_scaled)

# Min-max scale the raw score to make it user friendly [0, 100]
min_raw = df_features['iforest_raw_score'].min()
max_raw = df_features['iforest_raw_score'].max()
df_features['IForestScore'] = ((df_features['iforest_raw_score'] - min_raw) / (max_raw - min_raw + 1e-6)) * 100

print(f"Isolation Forest score range: {df_features['IForestScore'].min():.1f} to {df_features['IForestScore'].max():.1f}")

### 2. DBSCAN Clustering for Fleet Profiling

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups together points that are close to each other based on a distance measure. Points that lie alone in low-density regions are marked as outliers (cluster ID = -1).

In [ ]:
# Train DBSCAN
db = DBSCAN(eps=1.5, min_samples=10)
db_clusters = db.fit_predict(X_scaled)
df_features['dbscan_cluster'] = db_clusters
df_features['IsDBSCANOutlier'] = (db_clusters == -1).astype(int)

num_clusters = len(set(db_clusters)) - (1 if -1 in db_clusters else 0)
num_outliers = (db_clusters == -1).sum()
print(f"DBSCAN found {num_clusters} dense clusters and {num_outliers} outliers.")

In [ ]:
# Combine Isolation Forest and DBSCAN scores into a composite Risk Score
df_features['RiskScore'] = (0.7 * df_features['IForestScore'] + 30.0 * df_features['IsDBSCANOutlier']).clip(0, 100)
print("Risk scores computed.")

## ACT — Prioritize and Investigate anomalies

Let's check the top anomalies sorted by their final `RiskScore`. We will filter out events where the RiskScore is below a threat-hunting threshold (e.g. 50).

In [ ]:
# Sort findings
ranked_findings = df_features.sort_values(by='RiskScore', ascending=False)

# Display top 15 highest-risk allocations
display_cols = [
    'Computer', 'InitiatingProcessFileName', 'TargetProcessFileName', 
    'ActionType', 'RequestedSize', 'RequestedProtect', 'ThreadStartAddress', 
    'UnmappedAddress', 'IForestScore', 'IsDBSCANOutlier', 'RiskScore'
]
top_anomalies = ranked_findings[display_cols].head(15)
top_anomalies

### Visualizing Anomalies

Let's visualize the results to understand where the outliers lie relative to standard process behaviors.
We will plot a scatter plot comparing the `RequestedSize` (log) against the initiating process `FleetRarity`, highlighting the anomalous events.

In [ ]:
plt.figure(figsize=(10, 6))
# Plot background points
bg_mask = df_features['RiskScore'] < 50
plt.scatter(
    df_features.loc[bg_mask, 'LogSize'], 
    df_features.loc[bg_mask, 'FleetRarity'], 
    c='lightgray', alpha=0.6, label='Normal allocations'
)
# Plot threat points
fg_mask = df_features['RiskScore'] >= 50
plt.scatter(
    df_features.loc[fg_mask, 'LogSize'], 
    df_features.loc[fg_mask, 'FleetRarity'], 
    c=df_features.loc[fg_mask, 'RiskScore'], cmap='Reds', edgecolors='black', s=100, label='Anomalous allocations'
)
plt.colorbar(label='Risk Score')
plt.xlabel('Requested Allocation Size (Log10)')
plt.ylabel('Process Fleet Rarity')
plt.title('Threat Hunting: Process Memory Allocations (Log Size vs Fleet Rarity)')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'anomaly_distribution.png')
plt.show()

In [ ]:
# Plot anomaly score distribution
plt.figure(figsize=(8, 4))
sns.histplot(df_features['RiskScore'], bins=30, kde=True, color='purple')
plt.axvline(x=50, color='red', linestyle='--', label='Hunt Threshold (RiskScore=50)')
plt.xlabel('Risk Score')
plt.ylabel('Count')
plt.title('Distribution of Process Memory Allocation Risk Scores')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'risk_score_distribution.png')
plt.show()

In [ ]:
# Save findings to output folder
output_file = OUTPUT_DIR / 'process_memory_anomalies.csv'
ranked_findings.to_csv(output_file, index=False)
print(f"Successfully wrote {len(ranked_findings):,} ranked findings to {output_file.relative_to(REPO_ROOT)}")

# Summarize findings for output
high_risk_findings = ranked_findings[ranked_findings['RiskScore'] >= 50]
print(f"Number of high-risk anomalies detected: {len(high_risk_findings)}")
for idx, row in high_risk_findings.iterrows():
    init_name = row['InitiatingProcessFileName'].split('\\')[-1]
    target_name = row['TargetProcessFileName'].split('\\')[-1]
    print(f"- {row['Computer']}: {init_name} -> {target_name} ({row['ActionType']}) | Size: {row['RequestedSize']:,} | Protect: {row['RequestedProtect']} | Unmapped Start: {row['UnmappedAddress']} | Risk Score: {row['RiskScore']:.1f}")

## KNOWLEDGE — Feedback Loop and Exclusions

### Fleet Tuning and Noise Reductions

In a typical production environment, you will find legitimate applications that allocate memory dynamically:
1. **JIT compilation engines** such as browsers (`chrome.exe`, `msedge.exe`) or runtime environments (`java.exe`, `w3wp.exe` under IIS). These frequently request `PAGE_EXECUTE_READWRITE` (RWX) allocations inside their own processes.
2. **Security tools** and host agents that inject hooks or load monitors into active processes.

To suppress this noise:
- Add rules to target only cross-process allocations (`IsRemote == 1`).
- Filter out self-allocations from known JIT processes unless the size is extremely rare or the thread start address is unmapped.
- Maintain a local exclusion file in `exclusions/excluded_values.conf` containing safe process names or paths.